# Module 8: Fine-tune or prompt?

**Project:** take one narrow task and solve it **three ways**: with a small model *as it is*, with a **larger model that you prompt with examples**, and with the **small model fine-tuned with LoRA**. Then compare them on **quality, cost and speed**, and write a recommendation a founder could act on.

**Done when:**
- you built a dataset of **at least 200 examples** for one narrow task and split it into train, validation and test,
- you fine-tuned a small model with **LoRA** and evaluated all three systems on the **same held-out test set**,
- you have a **comparison table** (quality, cost, latency) and a **written recommendation** that uses the numbers.

**How to use this notebook**
1. Turn on a GPU: *Runtime > Change runtime type > T4 GPU*, then save your own copy (*File > Save a copy in Drive*).
2. Run cells from top to bottom. Fill in every `YOUR CODE HERE`. The **Self-check** cells tell you if you got it right.
3. The first runs download models, which takes a few minutes. After that, everything is quick.
4. Stuck? Use the hint ladder: docs, then the hints below each step, then the course forum. Only after your own attempt, open the solution page.

## Step 0: Setup (given)

In [ ]:
import json, re, time, random, csv
from pathlib import Path
import numpy as np
import torch

try:
    import peft
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "peft"], check=True)
    import peft
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"      # the SMALL model we will fine-tune (about 0.5 billion parameters)
LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]   # the layers LoRA adapts

def seed_all(seed=0):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
seed_all(0)
print("Using device:", device)

## Step 1: Choose one narrow task and build the dataset

Fine-tuning pays off on **narrow, repeated tasks** where you need the **same output format every time**: classifying messages, extracting fields into JSON, rewriting text in a house style. It is a poor way to add large amounts of fresh knowledge (that is what RAG is for).

**Your task needs:**
- an **input** (text) and an **output** (text) for each example,
- **at least 200 examples** (more is better),
- outputs that follow a **fixed format**, ideally JSON such as `{"label": "...", "name": "..."}`.

**Option A (recommended): your own task.** Put the examples in a file `my_task.csv` with two columns, `input` and `output`, and upload it. Write the instruction for the model in `TASK_SYSTEM` below. Set `OUTPUT_IS_JSON = False` if your outputs are plain labels instead of JSON.

**Option B (demo):** set `USE_DEMO = True` to use a ready-made task. It reads short customer-support messages (with typos and distracting numbers) and must answer with `{"intent": ..., "order_id": ...}`.

In [ ]:
USE_DEMO = False
CSV_PATH = Path("my_task.csv")
OUTPUT_IS_JSON = True       # False if your outputs are plain text labels
EVAL_N = None               # evaluate on at most this many test examples (None = all). Lower it to save time.

# The instruction given to every model. Replace it if you use your own task.
TASK_SYSTEM = ('You extract structured data from customer messages. Reply with JSON only, in exactly this format: '
               '{"intent": "<refund|cancel_order|track_order|change_address|other>", "order_id": "<the 5-digit order number, or none>"}')

def make_demo_dataset(n=400, seed=0):
    """Customer-support messages -> a fixed JSON answer: {"intent": ..., "order_id": ...}."""
    rng = random.Random(seed)
    openers = {
        "refund": ["I want my money back for", "please refund", "I'd like a refund on", "can you refund", "I need a refund for"],
        "cancel_order": ["please cancel", "I want to cancel", "I changed my mind, cancel", "kindly stop and cancel", "cancel"],
        "track_order": ["where is", "can you track", "what is the status of", "any update on", "when will arrive"],
        "change_address": ["I need to change the delivery address for", "wrong address on", "please ship to a new address for", "update the shipping address of"],
    }
    others = ["hello, do you sell gift cards?", "what are your opening hours?", "thanks for the great service!",
              "how do I reset my password?", "do you ship abroad?", "is there a student discount?", "good morning, I have a general question"]
    id_forms = ["order #{i}", "my order {i}", "order number {i}", "#{i}", "order {i}", "ref {i}"]
    distract = ["I called 555-0142 yesterday.", "I paid $34.50 on March 3.", "My ticket is T-77.", "Phone: 416-555-0199.",
                "It was 2 items.", "This is the 3rd time I write.", "Delivery was promised for the 12th."]
    closers = ["Thanks!", "This is urgent.", "Please reply soon.", "Thank you.", "", "", ""]

    def typo(s):
        words = s.split()
        cand = [k for k, w in enumerate(words) if len(w) > 4 and not any(ch.isdigit() for ch in w)]
        if cand and rng.random() < 0.3:
            k = rng.choice(cand); w = words[k]; j = rng.randrange(1, len(w) - 1)
            words[k] = w[:j] + w[j + 1:] if rng.random() < 0.5 else w[:j] + w[j + 1] + w[j] + w[j + 2:]
        return " ".join(words)

    out, seen = [], set()
    while len(out) < n:
        if rng.random() < 0.15:
            text, intent, oid = rng.choice(others), "other", "none"
        else:
            intent = rng.choice(list(openers))
            op = rng.choice(openers[intent])
            if rng.random() < 0.15:
                oid, text = "none", f"{op} my order but I forgot the number."
            else:
                oid = str(rng.randint(10000, 99999))
                text = f"{op} " + rng.choice(id_forms).format(i=oid) + "."
        parts = [text]
        if rng.random() < 0.5: parts.append(rng.choice(distract))
        c = rng.choice(closers)
        if c: parts.append(c)
        msg = typo(" ".join(parts))
        if msg in seen: continue
        seen.add(msg)
        out.append({"input": msg, "output": json.dumps({"intent": intent, "order_id": oid})})
    return out


def load_examples():
    if USE_DEMO:
        return make_demo_dataset(400, seed=0)
    assert CSV_PATH.exists(), "my_task.csv not found. Upload it (columns: input, output) or set USE_DEMO = True."
    with open(CSV_PATH, newline="", encoding="utf-8") as f:
        rows = list(csv.DictReader(f))
    return [{"input": r["input"].strip(), "output": r["output"].strip()} for r in rows if r.get("input") and r.get("output")]

examples = load_examples()
random.Random(0).shuffle(examples)
n = len(examples); n_test = int(0.2 * n); n_val = int(0.1 * n)
test_set, val_set, train_set = examples[:n_test], examples[n_test:n_test + n_val], examples[n_test + n_val:]
eval_examples = test_set[:EVAL_N] if EVAL_N else test_set
print(f"{n} examples: train {len(train_set)}, validation {len(val_set)}, test {len(test_set)} (evaluating on {len(eval_examples)})")
for e in train_set[:3]:
    print("IN :", e["input"]); print("OUT:", e["output"], "\n")

### Self-check: dataset

In [ ]:
assert len(examples) >= 200, f"You have {len(examples)} examples. You need at least 200."
assert all(e["input"] and e["output"] for e in examples), "Every example needs a non-empty input and output."
assert len({e["input"] for e in examples}) == len(examples), "Some inputs appear twice. Remove the duplicates, or the same example could land in both train and test."
if OUTPUT_IS_JSON:
    for _e in examples:
        json.loads(_e["output"])        # raises an error if an output is not valid JSON
assert not ({e["input"] for e in train_set} & {e["input"] for e in test_set}), "Train and test must not share inputs."
assert len(test_set) >= 30, f"The test set has only {len(test_set)} examples. Add more data."
print(f"Dataset check passed: {len(train_set)} train / {len(val_set)} validation / {len(test_set)} test.")

## Step 2: How will we judge an answer?

Both models will answer in free text, so we need a fair way to score them. Write two functions:

**`normalize_output(text)`** turns an answer into a canonical form so that harmless differences do not count as errors:
1. If the text contains a JSON object (find the first `{ ... }` with `re.search(r"\{.*\}", text, re.S)`) and it parses, return it re-written with `json.dumps(obj, sort_keys=True)`. This ignores spacing, key order, and chatter or code fences around it.
2. Otherwise return the text in lowercase with all whitespace collapsed to single spaces and stripped.

**`is_correct(pred, gold)`** is `True` when the normalized prediction equals the normalized gold answer.

(The quality score is deliberately forgiving about *chatter around* the JSON. Strictness about the format is measured separately, by `format_valid`, given below.)

In [ ]:
def format_valid(text):
    """Strict format check: is the WHOLE answer a JSON object? (Always True for non-JSON tasks.)"""
    if not OUTPUT_IS_JSON:
        return True
    try:
        return isinstance(json.loads(text.strip()), dict)
    except Exception:
        return False

def normalize_output(text):
    """Canonical form of an answer, so that harmless differences do not count as errors."""
    # YOUR CODE HERE
    return text

def is_correct(pred, gold):
    """True if the prediction matches the gold answer after normalizing both."""
    # YOUR CODE HERE
    return False

### Self-check: scoring

In [ ]:
assert normalize_output('{"b": 2, "a": 1}') == normalize_output('{"a":1,"b":2}'), "Key order and spacing must not matter for JSON."
assert normalize_output('Here you go:\n```json\n{"a": 1}\n```') == normalize_output('{"a": 1}'), "Chatter or code fences around a JSON object must be ignored."
assert normalize_output("  Refund   Now ") == normalize_output("refund now"), "Plain text should be lowercased with whitespace collapsed."
assert is_correct('{"intent": "refund", "order_id": "1"}', '{"order_id":"1","intent":"refund"}') is True, "Same JSON in a different layout is correct."
assert is_correct('{"intent": "refund", "order_id": "2"}', '{"intent": "refund", "order_id": "1"}') is False, "A wrong value is wrong."
assert is_correct("cancel order", "Cancel  Order") is True and is_correct("refund", "cancel") is False, "Plain-text answers are compared after normalizing."
print("Scoring check passed.")

## Step 3: Measure a system fairly

Every system we test (small model, large model, fine-tuned model) will be wrapped as a **predict function**: it takes the input text and returns a dictionary `{"text": answer, "input_tokens": n, "output_tokens": m}`.

Write `evaluate_system(predict_fn, examples)`. For every example it must call `predict_fn(example["input"])` and **time the call** (`time.perf_counter()` before and after). It returns a dictionary with:
- `quality`: the fraction of examples where `is_correct(answer, gold output)`,
- `valid_rate`: the fraction of answers where `format_valid(answer)`,
- `latency_s`: the average seconds per call,
- `avg_in`, `avg_out`: the average input and output tokens,
- `n`: the number of examples.

In [ ]:
def evaluate_system(predict_fn, examples):
    """Run predict_fn on every example. Return a dict with quality, valid_rate, latency_s, avg_in, avg_out, n."""
    # YOUR CODE HERE
    return {"quality": 0.0, "valid_rate": 0.0, "latency_s": 0.0, "avg_in": 0.0, "avg_out": 0.0, "n": len(examples)}

### Self-check: evaluate_system

In [ ]:
_prev_json = OUTPUT_IS_JSON; OUTPUT_IS_JSON = True
try:
    _gold = [{"input": "a", "output": '{"x": 1}'}, {"input": "b", "output": '{"x": 2}'},
             {"input": "c", "output": '{"x": 3}'}, {"input": "d", "output": '{"x": 4}'}]
    _replies = {"a": '{"x":1}', "b": 'Sure! {"x": 2}', "c": '{"x": 9}', "d": '{"x": 0}'}
    def _fake(text):
        time.sleep(0.01)
        return {"text": _replies[text], "input_tokens": 100, "output_tokens": 10}
    _r = evaluate_system(_fake, _gold)
finally:
    OUTPUT_IS_JSON = _prev_json
assert abs(_r["quality"] - 0.5) < 1e-9, f"Quality should be 0.5 (answers a and b are correct) but is {_r['quality']}. Remember is_correct() ignores chatter around the JSON."
assert abs(_r["valid_rate"] - 0.75) < 1e-9, f"valid_rate should be 0.75 (only answer b has chatter around its JSON, so 3 of 4 are pure JSON) but is {_r['valid_rate']}. Use format_valid(), not is_correct()."
assert _r["avg_in"] == 100 and _r["avg_out"] == 10 and _r["n"] == 4, "avg_in, avg_out and n are wrong."
assert 0.01 <= _r["latency_s"] < 0.5, f"latency_s should be a little over 0.01 seconds but is {_r['latency_s']}. Time the call to predict_fn."
print("evaluate_system() check passed.")

## Step 4: The small model, as it is (given)

Load the small model and ask it to do the task with **only the instruction** (no examples, no training). This is the baseline you must beat. `small_generate` builds a chat prompt and lets the model write the answer.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def load_base():
    try:
        m = AutoModelForCausalLM.from_pretrained(BASE_MODEL, dtype=torch.float32)
    except TypeError:
        m = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.float32)
    return m.to(device)

model = load_base()
print(f"Loaded {BASE_MODEL}: {sum(p.numel() for p in model.parameters()) / 1e6:.0f} million parameters")

@torch.no_grad()
def small_generate(m, user_text, max_new_tokens=64):
    """Greedy answer from a model m (with the task instruction, no examples). Returns text and token counts."""
    messages = [{"role": "system", "content": TASK_SYSTEM}, {"role": "user", "content": user_text}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    enc = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(device)
    m.eval()
    out = m.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.pad_token_id)
    new_ids = out[0, enc["input_ids"].shape[1]:]
    return {"text": tokenizer.decode(new_ids, skip_special_tokens=True).strip(),
            "input_tokens": int(enc["input_ids"].shape[1]), "output_tokens": int(len(new_ids))}

for e in test_set[:3]:
    print("IN :", e["input"]); print("GOLD:", e["output"]); print("UNTUNED SMALL MODEL:", small_generate(model, e["input"])["text"][:150], "\n")

**Look at the three answers.** Is the format right? Is the content right? Write what you notice:

*Your notes:*

Now measure it on the test set. This cell is given.

In [ ]:
results = {}
results["small, untuned"] = evaluate_system(lambda t: small_generate(model, t), eval_examples)
r = results["small, untuned"]
print(f"small, untuned: quality {r['quality']:.0%}  valid format {r['valid_rate']:.0%}  latency {r['latency_s']:.2f}s  tokens {r['avg_in']:.0f} in / {r['avg_out']:.0f} out")

## Step 5: Prompting a larger model, with examples

The usual first move is **not** to train anything: use a stronger model and show it a few examples in the prompt (**few-shot prompting**). Write `build_fewshot_prompt(shots, input_text)`. It returns `(system, user)`:
- `system` is `TASK_SYSTEM`,
- `user` starts with the line `Examples:` and a blank line, then for every shot a block `Message: <input>` / `Answer: <output>` followed by a blank line, and ends with `Message: <input_text>` and a final line `Answer:` (open, for the model to complete).

*Hint:* `shots` is a list of example dictionaries with `"input"` and `"output"`.

In [ ]:
def build_fewshot_prompt(shots, input_text):
    """Return (system, user): the instruction, and a prompt with worked examples followed by the new message."""
    system = TASK_SYSTEM
    user = ""
    # YOUR CODE HERE
    return system, user

### Self-check: build_fewshot_prompt

In [ ]:
_shots = [{"input": "first in", "output": "first out"}, {"input": "second in", "output": "second out"}]
_s, _u = build_fewshot_prompt(_shots, "NEW MESSAGE")
assert _s == TASK_SYSTEM, "The system message must be TASK_SYSTEM."
assert _u.startswith("Examples:"), "The user message must start with 'Examples:'."
assert "Message: first in\nAnswer: first out" in _u and "Message: second in\nAnswer: second out" in _u, "Each shot must be written as 'Message: ...' then 'Answer: ...' on the next line."
assert _u.index("first out") < _u.index("second in") < _u.index("NEW MESSAGE"), "Order: the shots in order, then the new message."
assert _u.rstrip().endswith("Message: NEW MESSAGE\nAnswer:"), "End with 'Message: <new input>' and a last line 'Answer:' for the model to complete."
assert _u.count("Answer:") == len(_shots) + 1, "There should be one 'Answer:' per shot, plus the open one."
print("build_fewshot_prompt() check passed.")

Now connect a **larger model**. Like in Module 7, you choose with `LLM_PROVIDER`:

- **`"local"`**: a larger open model in Colab (default: Qwen 2.5 with 3 billion parameters, six times bigger than the small model). Free, needs the GPU.
- **`"anthropic"`**: a hosted Claude model through the API. Needs an API key in Colab's **Secrets** panel (name: `ANTHROPIC_API_KEY`). Costs a little money.

This cell is given.

In [ ]:
LLM_PROVIDER = "local"                          # "local" or "anthropic"
LARGE_LOCAL_MODEL = "Qwen/Qwen2.5-3B-Instruct"
API_MODEL = "claude-haiku-4-5-20251001"
_llm = {}

def generate(system, user, max_tokens=80):
    """Ask the LARGER model. Returns {"text": ..., "input_tokens": ..., "output_tokens": ...}."""
    if LLM_PROVIDER == "anthropic":
        if "client" not in _llm:
            import anthropic
            from google.colab import userdata
            _llm["client"] = anthropic.Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"))
        r = _llm["client"].messages.create(model=API_MODEL, max_tokens=max_tokens, system=system,
                                           messages=[{"role": "user", "content": user}])
        return {"text": r.content[0].text.strip(), "input_tokens": r.usage.input_tokens, "output_tokens": r.usage.output_tokens}
    if "pipe" not in _llm:
        from transformers import pipeline
        try:
            _llm["pipe"] = pipeline("text-generation", model=LARGE_LOCAL_MODEL, device_map="auto", dtype=torch.float16)
        except TypeError:
            _llm["pipe"] = pipeline("text-generation", model=LARGE_LOCAL_MODEL, device_map="auto", torch_dtype=torch.float16)
    pipe = _llm["pipe"]
    messages = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    out = pipe(messages, max_new_tokens=max_tokens, do_sample=False)
    text = out[0]["generated_text"][-1]["content"].strip()
    prompt_text = pipe.tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    return {"text": text, "input_tokens": len(pipe.tokenizer(prompt_text)["input_ids"]),
            "output_tokens": len(pipe.tokenizer(text)["input_ids"])}

shots = train_set[:5]            # five worked examples shown in every prompt
def predict_large(text):
    system, user = build_fewshot_prompt(shots, text)
    return generate(system, user)

print("Large model:", API_MODEL if LLM_PROVIDER == "anthropic" else LARGE_LOCAL_MODEL)
print("Example answer:", predict_large(test_set[0]["input"])["text"][:150])

In [ ]:
results["large, few-shot"] = evaluate_system(predict_large, eval_examples)
r = results["large, few-shot"]
print(f"large, few-shot: quality {r['quality']:.0%}  valid format {r['valid_rate']:.0%}  latency {r['latency_s']:.2f}s  tokens {r['avg_in']:.0f} in / {r['avg_out']:.0f} out")

**Notice** how many **input tokens** the few-shot prompt costs on *every single request*, because the examples are sent again and again. Does the larger model keep to the exact format? Write what you see:

*Your notes:*

## Step 6: Prepare the training data

Fine-tuning teaches by example, but there is a trap: if the model also learns to *repeat the prompt*, it wastes effort on text that is already given. The standard fix is to compute the loss **only on the answer**, by marking the prompt tokens with the label **-100** (PyTorch ignores them).

Write `to_chat(example)` and `tokenize_example(example, tokenizer, max_len)`:

**`to_chat(example)`** returns the list of three messages: a `system` message with `TASK_SYSTEM`, a `user` message with the input, and an `assistant` message with the output.

**`tokenize_example`** returns `{"input_ids": ..., "labels": ...}` (two lists of the same length):
1. `prompt_text = tokenizer.apply_chat_template(messages[:-1], tokenize=False, add_generation_prompt=True)` (everything except the assistant's answer),
2. tokenize the prompt and the completion **separately** (with `add_special_tokens=False`): the completion is `example["output"] + tokenizer.eos_token`,
3. `input_ids` = prompt ids + completion ids,
4. `labels` = `-100` for each prompt position, then the completion ids,
5. cut both to `max_len`.

In [ ]:
def to_chat(example):
    """The three chat messages (system, user, assistant) for one training example."""
    # YOUR CODE HERE
    return []

def tokenize_example(example, tokenizer, max_len=256):
    """Return {"input_ids": [...], "labels": [...]} where the labels ignore (-100) the prompt and learn only the answer."""
    # YOUR CODE HERE
    return {"input_ids": [], "labels": []}

### Self-check: training data

In [ ]:
_ex = train_set[0]
_m = to_chat(_ex)
assert [x["role"] for x in _m] == ["system", "user", "assistant"], "to_chat() must return the roles system, user, assistant in that order."
assert _m[0]["content"] == TASK_SYSTEM and _m[1]["content"] == _ex["input"] and _m[2]["content"] == _ex["output"], "The message contents are wrong."
_t = tokenize_example(_ex, tokenizer, max_len=1000)
assert len(_t["input_ids"]) == len(_t["labels"]), "input_ids and labels must have the same length."
_learn = [l for l in _t["labels"] if l != -100]
_expected = tokenizer(_ex["output"] + tokenizer.eos_token, add_special_tokens=False)["input_ids"]
assert _learn == _expected, "The positions that are NOT -100 must be exactly the answer followed by the end-of-text token."
_p = len(_t["labels"]) - len(_learn)
assert _p > 0 and all(l == -100 for l in _t["labels"][:_p]), "Every prompt position must have the label -100."
assert _t["input_ids"][_p:] == _learn, "In the answer part, labels must equal input_ids."
_short = tokenize_example(_ex, tokenizer, max_len=10)
assert len(_short["input_ids"]) == 10 and len(_short["labels"]) == 10, "Both lists must be cut to max_len."
print(f"Training data check passed. Example: {_p} prompt tokens (ignored) + {len(_learn)} answer tokens (learned).")

## Step 7: LoRA

Training all 500 million weights would need a lot of memory and produce a whole new copy of the model. **LoRA** (low-rank adaptation) **freezes** the original weights and adds a pair of **small matrices** next to each chosen layer. Only the small matrices are trained. The result is a tiny **adapter file** (a few megabytes) that you plug into the unchanged base model.

Write `make_lora_model(model, r, alpha, dropout)`. It creates a `LoraConfig` with `r=r`, `lora_alpha=alpha`, `lora_dropout=dropout`, `target_modules=LORA_TARGETS` and `task_type="CAUSAL_LM"`, and returns `get_peft_model(model, config)`.

- `r` is the **rank**: the size of the small matrices. Bigger means more capacity and more trainable weights.
- `alpha` scales the adapter's contribution (a common choice is twice `r`).

In [ ]:
def make_lora_model(model, r=16, alpha=32, dropout=0.05):
    """Wrap a model with LoRA adapters and return the wrapped (PEFT) model."""
    # YOUR CODE HERE
    return model

### Self-check: make_lora_model

In [ ]:
from transformers import LlamaConfig, LlamaForCausalLM
_tiny = LlamaForCausalLM(LlamaConfig(hidden_size=32, intermediate_size=64, num_hidden_layers=2, num_attention_heads=4, vocab_size=100))
_total_before = sum(p.numel() for p in _tiny.parameters())
_wrapped = make_lora_model(_tiny, r=4, alpha=8, dropout=0.0)
_trainable = {n: p for n, p in _wrapped.named_parameters() if p.requires_grad}
assert _trainable, "No parameters are trainable. Did you return the wrapped model from get_peft_model()?"
assert all("lora_" in n for n in _trainable), "Only the LoRA matrices should be trainable, but these are too: " + str([n for n in _trainable if "lora_" not in n][:3])
_a = [p for n, p in _trainable.items() if "lora_A" in n]
assert _a and all(p.shape[0] == 4 for p in _a), "The LoRA matrices should have rank r=4. Pass r to LoraConfig."
_tr = sum(p.numel() for p in _trainable.values())
assert _tr < 0.2 * _total_before, f"LoRA should train far fewer weights than the model has, but trains {_tr:,} of {_total_before:,}."
print(f"make_lora_model() check passed: LoRA trains {_tr:,} weights on a model with {_total_before:,} ({100 * _tr / _total_before:.1f}%).")
del _tiny, _wrapped

Now train. The training loop is given: it is the loop you know from Module 3, with the answer-only labels. **This is the step that makes your model special.**

In [ ]:
def collate(batch, pad_id):
    longest = max(len(b["input_ids"]) for b in batch)
    ids = [b["input_ids"] + [pad_id] * (longest - len(b["input_ids"])) for b in batch]
    labels = [b["labels"] + [-100] * (longest - len(b["labels"])) for b in batch]
    mask = [[1] * len(b["input_ids"]) + [0] * (longest - len(b["input_ids"])) for b in batch]
    return {"input_ids": torch.tensor(ids), "labels": torch.tensor(labels), "attention_mask": torch.tensor(mask)}

def train_lora(peft_model, examples, epochs=3, lr=2e-4, batch_size=8, max_len=256):
    data = [tokenize_example(e, tokenizer, max_len) for e in examples]
    opt = torch.optim.AdamW([p for p in peft_model.parameters() if p.requires_grad], lr=lr)
    peft_model.train(); losses = []
    for epoch in range(epochs):
        random.shuffle(data); total, count = 0.0, 0
        for i in range(0, len(data), batch_size):
            batch = {k: v.to(device) for k, v in collate(data[i:i + batch_size], tokenizer.pad_token_id).items()}
            loss = peft_model(**batch).loss
            loss.backward(); opt.step(); opt.zero_grad()
            losses.append(loss.item()); total += loss.item(); count += 1
        print(f"epoch {epoch + 1}/{epochs}: average loss {total / count:.3f}")
    peft_model.eval()
    return losses

seed_all(0)
model = make_lora_model(model, r=16, alpha=32, dropout=0.05)
model.print_trainable_parameters()
t0 = time.time()
losses = train_lora(model, train_set, epochs=3, lr=2e-4)
train_seconds = time.time() - t0
print(f"Training took {train_seconds:.0f} seconds ({len(train_set)} examples, 3 epochs).")
model.save_pretrained("lora_adapter")
adapter_mb = sum(f.stat().st_size for f in Path("lora_adapter").rglob("*") if f.is_file()) / 1e6
print(f"The whole adapter is only {adapter_mb:.1f} MB.")

### Self-check: training

In [ ]:
assert len(losses) >= 5, "The training loop did not run."
_k = max(2, len(losses) // 5)
assert np.mean(losses[-_k:]) < np.mean(losses[:_k]), "The loss did not go down. Check the labels (answer only), the learning rate, and the data."
assert adapter_mb < 200, "The adapter should be small (a few MB). Did you save the whole model instead of the adapter?"
print(f"Training check passed. Loss went from about {np.mean(losses[:_k]):.2f} to about {np.mean(losses[-_k:]):.2f}.")

## Step 8: Evaluate the fine-tuned model (given)

The same test set, the same scoring. The fine-tuned model gets **only the short instruction**: no examples in the prompt. Compare it with the untuned model, with the same weights underneath: the PEFT model can switch its adapter off to show the original.

In [ ]:
with model.disable_adapter():
    print("ADAPTER OFF (the original small model):", small_generate(model, test_set[0]["input"])["text"][:120])
print("ADAPTER ON  (fine-tuned):             ", small_generate(model, test_set[0]["input"])["text"][:120])
print("GOLD:                                 ", test_set[0]["output"])

results["small + LoRA"] = evaluate_system(lambda t: small_generate(model, t), eval_examples)
for name, r in results.items():
    print(f"{name:18s} quality {r['quality']:.0%}  valid format {r['valid_rate']:.0%}  latency {r['latency_s']:.2f}s  tokens {r['avg_in']:.0f} in / {r['avg_out']:.0f} out")

### Self-check: fine-tuning helped

In [ ]:
_ft, _base = results["small + LoRA"], results["small, untuned"]
assert _ft["valid_rate"] >= 0.9, f"Only {_ft['valid_rate']:.0%} of the fine-tuned answers are in the exact format. Train longer, or check that the labels teach the format."
assert _ft["quality"] >= _base["quality"] + 0.2, (f"The fine-tuned model ({_ft['quality']:.0%}) is not clearly better than the untuned one ({_base['quality']:.0%}). "
    "Try more epochs, a higher learning rate, or check the data and labels.")
print(f"Fine-tuning check passed: quality {_base['quality']:.0%} -> {_ft['quality']:.0%}, valid format {_base['valid_rate']:.0%} -> {_ft['valid_rate']:.0%}.")

## Step 9: What does each option cost?

Quality is only one axis. A startup also pays for **money** and **waiting time**. You need three small formulas. Write them:

**`api_cost_per_1k(avg_in, avg_out, price_in_per_m, price_out_per_m)`**: dollars per **1,000 requests** when you pay per token (prices are per **million** tokens).

**`gpu_cost_per_1k(latency_s, gpu_dollars_per_hour)`**: dollars per **1,000 requests** when you run the model yourself on a rented GPU. One request occupies the GPU for `latency_s` seconds, and an hour has 3,600 seconds.

**`break_even_requests(one_time_cost, cost_per_1k_baseline, cost_per_1k_new)`**: the number of requests after which paying a **one-time cost** (such as training) for a cheaper option has paid off. The saving per request is `(baseline - new) / 1000`. If the new option is not cheaper, return `float("inf")`.

In [ ]:
def api_cost_per_1k(avg_in, avg_out, price_in_per_m, price_out_per_m):
    """Dollars per 1,000 requests when paying per token."""
    # YOUR CODE HERE
    return 0.0

def gpu_cost_per_1k(latency_s, gpu_dollars_per_hour):
    """Dollars per 1,000 requests on a GPU you rent by the hour (one request at a time)."""
    # YOUR CODE HERE
    return 0.0

def break_even_requests(one_time_cost, cost_per_1k_baseline, cost_per_1k_new):
    """Number of requests after which a one-time cost pays for itself (infinity if the new option is not cheaper)."""
    # YOUR CODE HERE
    return 0.0

### Self-check: cost formulas

In [ ]:
assert abs(api_cost_per_1k(500, 20, 1.0, 5.0) - 0.6) < 1e-9, f"500 input and 20 output tokens at 1.0 and 5.0 dollars per million cost 0.0006 per request, so 0.6 per 1,000. You got {api_cost_per_1k(500, 20, 1.0, 5.0)}."
assert abs(gpu_cost_per_1k(0.2, 0.5) - 1000 * 0.2 / 3600 * 0.5) < 1e-9 and abs(gpu_cost_per_1k(0.2, 0.5) - 0.027778) < 1e-5, "0.2 seconds on a 0.50 dollars per hour GPU is about 0.0278 dollars per 1,000 requests."
assert abs(break_even_requests(5.0, 2.0, 0.5) - 3333.333) < 0.01, f"A one-time cost of 5 dollars that saves (2.0 - 0.5) / 1000 per request pays off after 3,333 requests. You got {break_even_requests(5.0, 2.0, 0.5)}."
assert break_even_requests(5.0, 0.5, 2.0) == float("inf") and break_even_requests(5.0, 1.0, 1.0) == float("inf"), "If the new option is not cheaper, it never pays off: return float('inf')."
print("Cost formulas check passed.")

Now set your prices and build the **comparison table**. The numbers below are **illustrative placeholders**: replace them with your provider's **current** prices (they change). A free Colab GPU costs nothing, but a rented one does, so use a realistic hourly price to see what you would pay in production. This cell is given.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

PRICE_IN_PER_M = 1.0          # dollars per million input tokens (if the large model is a paid API). PLACEHOLDER: look up the real price
PRICE_OUT_PER_M = 5.0         # dollars per million output tokens. PLACEHOLDER
GPU_DOLLARS_PER_HOUR = 0.50   # what renting a GPU for production might cost per hour. PLACEHOLDER

SYSTEM_KINDS = {"small, untuned": "gpu", "small + LoRA": "gpu",
                "large, few-shot": "api" if LLM_PROVIDER == "anthropic" else "gpu"}
rows = []
for name, r in results.items():
    per1k = (api_cost_per_1k(r["avg_in"], r["avg_out"], PRICE_IN_PER_M, PRICE_OUT_PER_M) if SYSTEM_KINDS[name] == "api"
             else gpu_cost_per_1k(r["latency_s"], GPU_DOLLARS_PER_HOUR))
    rows.append({"system": name, "quality": r["quality"], "valid format": r["valid_rate"], "latency (s)": r["latency_s"],
                 "tokens in": r["avg_in"], "tokens out": r["avg_out"], "cost per 1,000 requests ($)": per1k})
table = pd.DataFrame(rows)
display(table.round(4))

one_time = train_seconds / 3600 * GPU_DOLLARS_PER_HOUR
big, small = table.set_index("system").loc["large, few-shot", "cost per 1,000 requests ($)"], table.set_index("system").loc["small + LoRA", "cost per 1,000 requests ($)"]
be = break_even_requests(one_time, big, small)
print(f"One-time cost of the fine-tuning run: ${one_time:.4f}")
print(f"Fine-tuned small vs few-shot large: ${small:.4f} vs ${big:.4f} per 1,000 requests -> break-even after {be:,.0f} requests" if be != float("inf")
      else "The fine-tuned model is not cheaper than the large model in this setup, so fine-tuning never pays off on cost alone.")

fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for ax, col, title in zip(axes, ["quality", "cost per 1,000 requests ($)", "latency (s)"], ["Quality (higher is better)", "Cost per 1,000 requests (lower)", "Latency, seconds (lower)"]):
    ax.bar(table["system"], table[col]); ax.set_title(title); ax.tick_params(axis="x", rotation=25)
plt.tight_layout(); plt.show()

## Step 10: Your recommendation

Imagine you are advising the founder of a startup that will get **100,000 of these requests per day**. Which option should they choose: **the small model, the prompted large model, or the fine-tuned small model**? Write your recommendation as a short paragraph (at least 4 sentences) in the variable `RECOMMENDATION`. It must:
- name the option you recommend,
- use **numbers from your table** for **quality**, **cost** and **latency**,
- say when the answer would change (for example, a different volume, a task that changes often, or a need for privacy).

In [ ]:
RECOMMENDATION = """
YOUR CODE HERE: write your recommendation here.
"""

### Self-check: recommendation

In [ ]:
_text = RECOMMENDATION.lower()
assert "your code here" not in _text and len(RECOMMENDATION.strip()) >= 250, "Write a real recommendation of at least four sentences (about 250 characters or more)."
_axes = {"quality": ("quality", "accuracy", "correct"), "cost": ("cost", "price", "$", "cheap", "expensive"), "latency": ("latency", "speed", "fast", "slow", "seconds", "response time")}
_missing = [axis for axis, words in _axes.items() if not any(w in _text for w in words)]
assert not _missing, f"Your recommendation must discuss all three axes, and does not mention: {_missing}. Use the words quality, cost and latency (or speed)."
assert any(w in _text for w in ("fine-tun", "prompt", "untuned", "few-shot", "lora")), "Name the option you recommend (fine-tuned, prompted, or untuned small model)."
assert any(ch.isdigit() for ch in RECOMMENDATION), "Use numbers from your table to back up the recommendation."
table.to_csv("comparison_table.csv", index=False)
print("Recommendation check passed. Your table was saved as comparison_table.csv.")

## Stretch challenge (optional): quality versus data

How many training examples do you really need? Train fresh adapters on the first **25, 50, 100 and all** of your training examples (rebuild the base model with `load_base()` each time, and use the same test set). Plot **quality against the number of examples**, and draw a horizontal line at the quality of the few-shot large model. Where does the fine-tuned model overtake it? Is labelling more data worth it beyond that point?

*Your notes:*

## Reflection and submission

Write three lines in your portfolio repo's `module-08/README.md`:
1. Your task, and the three systems you compared
2. Your comparison table (paste it) and which system you recommend
3. What surprised you, and when your recommendation would change

Then download this notebook (*File > Download > .ipynb*), push it to `module-08` in your portfolio repo together with `comparison_table.csv` (do **not** push private data or API keys), and submit the link in Moodle.